# EDA Inicial — Cancelaciones de reservas hoteleras

- **Proyecto:** Práctica Final · Machine Learning y Deep Learning
- **Integrantes:** Juan Martínez Fraile · Otra persona la que sea
- **Máster:** Inteligencia Artificial, Cloud Computing y DevOps · PontIA.tech
- **Fase:** Análisis Exploratorio de Datos

---

## Objetivo

Realizar un análisis exploratorio del dataset de reservas hoteleras para:

1. Comprender la estructura y calidad de los datos.
2. Identificar problemas de modelado (data leakage, nulos, outliers, desbalanceo).
3. Justificar las decisiones de preprocesamiento y modelado de fases posteriores.
4. Seleccionar la métrica principal de evaluación.

## Temas a tratar

0. Configuración del notebook
1. Carga inicial y vista general de los datos
2. Análisis de la variable objetivo (`is_canceled`)
3. Estadísticas de variables numéricas
4. Estadísticas de variables categóricas
5. Análisis de nulos y valores especiales
6. Detección de data leakage
7. Análisis bivariado y correlaciones
8. Resumen de decisiones para la fase de preprocesamiento

## 0. Configuración del notebook

Importamos las librerías necesarias y definimos constantes/funciones que usaremos a lo largo del análisis.

### Importación de librerías

In [1]:
"""Importación de librerías necesarias para el análisis exploratorio."""

# Standard library
from pathlib import Path

# Análisis y manipulación de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# Configuración global de visualización
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Librerías importadas correctamente.")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Librerías importadas correctamente.
pandas: 2.2.3
numpy: 1.26.4


### Definición de constantes

In [2]:
"""Constantes utilizadas a lo largo del análisis exploratorio."""

# === Rutas ===
# Path(__vsc_ipynb_file__) no existe; usamos la raíz del proyecto como referencia.
# Como el notebook está en notebooks/exploracion/, subimos dos niveles a la raíz.
PATH_PROYECTO = Path.cwd().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
PATH_DATA_RAW = PATH_PROYECTO / "data" / "raw"
PATH_DATA_PROCESSED = PATH_PROYECTO / "data" / "processed"
PATH_OUTPUTS = PATH_PROYECTO / "outputs"

PATH_DATASET = PATH_DATA_RAW / "dataset_practica_final.csv"

# === Configuración del análisis ===
SEED = 42  # Semilla para reproducibilidad
COLOR_PALETTE = ["#1f77b4", "#ff7f0e"]  # Azul (no cancelado) / Naranja (cancelado)
TARGET_COLUMN = "is_canceled"

# Verificación visual
print(f"Raíz del proyecto:  {PATH_PROYECTO}")
print(f"Dataset:            {PATH_DATASET}")
print(f"¿Dataset existe?:   {PATH_DATASET.exists()}")
print(f"Tamaño del archivo: {PATH_DATASET.stat().st_size / 1024 / 1024:.2f} MB")

Raíz del proyecto:  c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml
Dataset:            c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml\data\raw\dataset_practica_final.csv
¿Dataset existe?:   True
Tamaño del archivo: 16.07 MB


### Definición de funciones

In [3]:
"""Funciones de ayuda reutilizables en todo el análisis exploratorio."""

from typing import Optional


def resumen_df(df: pd.DataFrame) -> pd.DataFrame:
    """Devuelve un resumen compacto de cada columna de un DataFrame.

    Para cada columna calcula tipo, número de valores únicos, número y porcentaje
    de nulos, y una muestra de valores. Es la primera vista útil tras cargar
    cualquier dataset nuevo.

    Args:
        df (pd.DataFrame): DataFrame a inspeccionar.

    Returns:
        pd.DataFrame: Tabla con una fila por columna del DataFrame original.

    Example:
        >>> resumen_df(df).head()
    """
    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_unicos": df.nunique(),
        "n_nulos": df.isnull().sum(),
        "pct_nulos": (df.isnull().sum() / len(df) * 100).round(2),
        "ejemplos": [df[col].dropna().unique()[:3].tolist() for col in df.columns],
    })
    return resumen.sort_values("pct_nulos", ascending=False)


def plot_distribucion_categorica(
    df: pd.DataFrame,
    columna: str,
    hue: Optional[str] = None,
    top_n: Optional[int] = None,
    figsize: tuple = (10, 5),
    title: Optional[str] = None,
) -> None:
    """Visualiza la distribución de una variable categórica con conteos.

    Útil para entender la composición de variables como `hotel`, `market_segment`
    o `country`. Si se pasa `hue`, descompone cada barra por la variable indicada
    (típicamente la variable objetivo).

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        columna (str): Nombre de la columna categórica a visualizar.
        hue (Optional[str]): Variable secundaria para descomponer (ej. el target).
        top_n (Optional[int]): Si se indica, muestra solo las `top_n` categorías más frecuentes.
        figsize (tuple): Tamaño de la figura (ancho, alto).
        title (Optional[str]): Título personalizado. Por defecto se genera automáticamente.

    Returns:
        None: La función dibuja directamente con matplotlib/seaborn.
    """
    plt.figure(figsize=figsize)

    # Filtrar a top_n si aplica
    if top_n is not None:
        top_categorias = df[columna].value_counts().head(top_n).index
        df_plot = df[df[columna].isin(top_categorias)]
        orden = top_categorias
    else:
        df_plot = df
        orden = df[columna].value_counts().index

    sns.countplot(
        data=df_plot,
        y=columna,
        hue=hue,
        order=orden,
        palette=COLOR_PALETTE if hue == TARGET_COLUMN else "deep",
    )

    if title is None:
        title = f"Distribución de '{columna}'"
        if hue:
            title += f" por '{hue}'"
        if top_n:
            title += f" (top {top_n})"

    plt.title(title)
    plt.xlabel("Número de reservas")
    plt.tight_layout()
    plt.show()


def plot_distribucion_numerica(
    df: pd.DataFrame,
    columna: str,
    hue: Optional[str] = None,
    bins: int = 50,
    figsize: tuple = (12, 4),
) -> None:
    """Visualiza histograma + boxplot de una variable numérica.

    Combina dos vistas complementarias: el histograma muestra la forma de la distribución,
    el boxplot muestra mediana, cuartiles y outliers. Si se pasa `hue`, separa por clases.

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        columna (str): Nombre de la columna numérica.
        hue (Optional[str]): Variable categórica para separar las distribuciones.
        bins (int): Número de bins del histograma.
        figsize (tuple): Tamaño de la figura.

    Returns:
        None
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Histograma
    sns.histplot(
        data=df,
        x=columna,
        hue=hue,
        bins=bins,
        ax=axes[0],
        palette=COLOR_PALETTE if hue == TARGET_COLUMN else None,
        kde=True,
    )
    axes[0].set_title(f"Histograma de '{columna}'")

    # Boxplot
    if hue:
        sns.boxplot(
            data=df,
            x=hue,
            y=columna,
            ax=axes[1],
            palette=COLOR_PALETTE if hue == TARGET_COLUMN else None,
        )
        axes[1].set_title(f"Boxplot de '{columna}' por '{hue}'")
    else:
        sns.boxplot(data=df, x=columna, ax=axes[1])
        axes[1].set_title(f"Boxplot de '{columna}'")

    plt.tight_layout()
    plt.show()


print("Funciones helper definidas correctamente.")

Funciones helper definidas correctamente.
